In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
path = '/content/drive/MyDrive/churnlens/data'
print(os.listdir(path))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/churnlens/data'

In [ ]:
# ============================================
# PHASE 1 — DATA ENGINEERING
# ChurnLens Project
# ============================================

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# ============================================
# STEP 1 — SET PATH
# ============================================
BASE_PATH = '/content/drive/MyDrive/churnlens/data/raw/'

# ============================================
# STEP 2 — LOAD ALL 9 OLIST TABLES
# ============================================
customers   = pd.read_csv(BASE_PATH + 'olist_customers_dataset.csv')
orders      = pd.read_csv(BASE_PATH + 'olist_orders_dataset.csv')
order_items = pd.read_csv(BASE_PATH + 'olist_order_items_dataset.csv')
payments    = pd.read_csv(BASE_PATH + 'olist_order_payments_dataset.csv')
reviews     = pd.read_csv(BASE_PATH + 'olist_order_reviews_dataset.csv')
products    = pd.read_csv(BASE_PATH + 'olist_products_dataset.csv')
sellers     = pd.read_csv(BASE_PATH + 'olist_sellers_dataset.csv')
geo         = pd.read_csv(BASE_PATH + 'olist_geolocation_dataset.csv')
category    = pd.read_csv(BASE_PATH + 'product_category_name_translation.csv')

print("✅ All 9 tables loaded successfully")
print(f"Orders: {orders.shape}")
print(f"Customers: {customers.shape}")
print(f"Order Items: {order_items.shape}")
print(f"Payments: {payments.shape}")
print(f"Reviews: {reviews.shape}")
print(f"Products: {products.shape}")
print(f"Sellers: {sellers.shape}")
print(f"Geolocation: {geo.shape}")
print(f"Category Translation: {category.shape}")

In [ ]:
# ============================================
# STEP 3 — MERGE ALL TABLES
# ============================================

# Merge 1: orders + customers
df = orders.merge(customers, on='customer_id', how='left')
print(f"After orders + customers: {df.shape}")

# Merge 2: + order_items
df = df.merge(order_items, on='order_id', how='left')
print(f"After + order_items: {df.shape}")

# Merge 3: + products
df = df.merge(products, on='product_id', how='left')
print(f"After + products: {df.shape}")

# Merge 4: + category translation
df = df.merge(category, on='product_category_name', how='left')
print(f"After + category translation: {df.shape}")

# Merge 5: + payments
df = df.merge(payments, on='order_id', how='left')
print(f"After + payments: {df.shape}")

# Merge 6: + reviews
df = df.merge(reviews, on='order_id', how='left')
print(f"After + reviews: {df.shape}")

# Merge 7: + sellers
df = df.merge(sellers, on='seller_id', how='left')
print(f"After + sellers: {df.shape}")

print("\n✅ All tables merged successfully")
print(f"Final shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# ============================================
# STEP 4 — DATA CLEANING
# ============================================

# Check null values before cleaning
print("Null values before cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Fix 1 — Fill missing product category
df['product_category_name_english'].fillna('unknown', inplace=True)

# Fix 2 — Fill missing review comments
df['review_comment_title'].fillna('no title', inplace=True)
df['review_comment_message'].fillna('no review', inplace=True)

# Fix 3 — Convert date columns to datetime
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

# Fix 4 — Drop columns we don't need
drop_cols = [
    'product_category_name',
    'product_name_lenght',
    'product_description_lenght',
    'product_photos_qty',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm',
    'customer_zip_code_prefix',
    'seller_zip_code_prefix',
    'payment_sequential',
    'review_id',
    'review_creation_date',
    'review_answer_timestamp',
    'shipping_limit_date',
    'order_item_id'
]
df.drop(columns=drop_cols, inplace=True)

# Check null values after cleaning
print("\nNull values after cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print(f"\n✅ Cleaning done")
print(f"Final shape: {df.shape}")
print(f"Remaining columns: {list(df.columns)}")

In [ ]:
# ============================================
# STEP 5 — HANDLE REMAINING NULLS + SAVE
# ============================================

# Fix date nulls — fill with median
df['order_approved_at'].fillna(df['order_approved_at'].median(), inplace=True)
df['order_delivered_carrier_date'].fillna(df['order_delivered_carrier_date'].median(), inplace=True)
df['order_delivered_customer_date'].fillna(df['order_delivered_customer_date'].median(), inplace=True)

# Fix product/seller nulls — fill with unknown/0
df['product_id'].fillna('unknown', inplace=True)
df['seller_id'].fillna('unknown', inplace=True)
df['seller_city'].fillna('unknown', inplace=True)
df['seller_state'].fillna('unknown', inplace=True)

# Fix price/freight nulls — fill with median
df['price'].fillna(df['price'].median(), inplace=True)
df['freight_value'].fillna(df['freight_value'].median(), inplace=True)

# Fix payment nulls — fill with mode
df['payment_type'].fillna(df['payment_type'].mode()[0], inplace=True)
df['payment_installments'].fillna(df['payment_installments'].mode()[0], inplace=True)
df['payment_value'].fillna(df['payment_value'].median(), inplace=True)

# Fix review score nulls — fill with median
df['review_score'].fillna(df['review_score'].median(), inplace=True)

# Final null check
print("Null values after final fix:")
remaining = df.isnull().sum()[df.isnull().sum() > 0]
if len(remaining) == 0:
    print("✅ Zero nulls remaining")
else:
    print(remaining)

# Save cleaned dataframe to Drive
SAVE_PATH = '/content/drive/MyDrive/churnlens/data/processed/'
os.makedirs(SAVE_PATH, exist_ok=True)
df.to_csv(SAVE_PATH + 'olist_cleaned.csv', index=False)

print(f"\n✅ Cleaned data saved to Drive")
print(f"Final shape: {df.shape}")
print(df.head(3))

Phase 2 — Churn Label Creation

In [ ]:
# ============================================
# PHASE 2 — CHURN LABEL CREATION
# ============================================

import pandas as pd
import numpy as np
import os

# ============================================
# STEP 1 — LOAD CLEANED DATA
# ============================================
SAVE_PATH = '/content/drive/MyDrive/churnlens/data/processed/'
df = pd.read_csv(SAVE_PATH + 'olist_cleaned.csv', parse_dates=['order_purchase_timestamp'])

print(f"✅ Cleaned data loaded: {df.shape}")

# ============================================
# STEP 2 — GET LAST PURCHASE PER CUSTOMER
# ============================================
# Always use customer_unique_id — not customer_id
# customer_id changes per order, unique_id is real identity

customer_orders = df.groupby('customer_unique_id').agg(
    total_orders        = ('order_id', 'nunique'),
    last_purchase_date  = ('order_purchase_timestamp', 'max'),
    first_purchase_date = ('order_purchase_timestamp', 'min')
).reset_index()

print(f"\n✅ Unique customers: {len(customer_orders)}")
print(customer_orders.head(3))

# ============================================
# STEP 3 — DEFINE CHURN LABEL
# ============================================
# Dataset ends at: 2018-08-29 (last order date)
# Churn = bought once AND last order > 180 days before end date

dataset_end_date = df['order_purchase_timestamp'].max()
print(f"\nDataset end date: {dataset_end_date}")

# Days since last purchase
customer_orders['days_since_last_purchase'] = (
    dataset_end_date - customer_orders['last_purchase_date']
).dt.days

# Churn label
# 1 = churned, 0 = retained
customer_orders['churn'] = (
    (customer_orders['total_orders'] == 1) &
    (customer_orders['days_since_last_purchase'] > 180)
).astype(int)

# ============================================
# STEP 4 — CHECK CLASS DISTRIBUTION
# ============================================
churn_counts = customer_orders['churn'].value_counts()
churn_pct    = customer_orders['churn'].value_counts(normalize=True) * 100

print(f"\nChurn Label Distribution:")
print(f"Retained (0): {churn_counts[0]} ({churn_pct[0]:.1f}%)")
print(f"Churned  (1): {churn_counts[1]} ({churn_pct[1]:.1f}%)")

# ============================================
# STEP 5 — SAVE CHURN LABELS
# ============================================
customer_orders.to_csv(SAVE_PATH + 'churn_labels.csv', index=False)
print(f"\n✅ Churn labels saved to Drive")
print(f"Shape: {customer_orders.shape}")
print(customer_orders.head(5))

Phase 3 — RFM Feature Engineering

In [ ]:
# ============================================
# PHASE 3 — RFM FEATURE ENGINEERING
# ============================================

import pandas as pd
import numpy as np
import os

# ============================================
# STEP 1 — LOAD BOTH FILES
# ============================================
SAVE_PATH = '/content/drive/MyDrive/churnlens/data/processed/'

df           = pd.read_csv(SAVE_PATH + 'olist_cleaned.csv',
                           parse_dates=['order_purchase_timestamp',
                                        'order_delivered_customer_date',
                                        'order_estimated_delivery_date'])
churn_labels = pd.read_csv(SAVE_PATH + 'churn_labels.csv')

print(f"✅ Cleaned data loaded:  {df.shape}")
print(f"✅ Churn labels loaded:  {churn_labels.shape}")

# ============================================
# STEP 2 — BUILD RFM FEATURES PER CUSTOMER
# ============================================
dataset_end_date = df['order_purchase_timestamp'].max()

rfm = df.groupby('customer_unique_id').agg(

    # RECENCY — days since last order
    recency = ('order_purchase_timestamp',
               lambda x: (dataset_end_date - x.max()).days),

    # FREQUENCY — total unique orders
    frequency = ('order_id', 'nunique'),

    # MONETARY — total amount spent
    monetary = ('payment_value', 'sum'),

).reset_index()

print(f"\n✅ RFM base built: {rfm.shape}")

# ============================================
# STEP 3 — EXTRA FEATURES
# ============================================
extra = df.groupby('customer_unique_id').agg(

    # Average review score
    avg_review_score = ('review_score', 'mean'),

    # Average freight value
    avg_freight_value = ('freight_value', 'mean'),

    # Average payment installments
    avg_installments = ('payment_installments', 'mean'),

    # Total product categories bought
    total_categories = ('product_category_name_english', 'nunique'),

    # Most used payment type
    preferred_payment = ('payment_type', lambda x: x.mode()[0]),

).reset_index()

print(f"✅ Extra features built: {extra.shape}")

# ============================================
# STEP 4 — DELIVERY DELAY FEATURE
# ============================================
df['delivery_delay_days'] = (
    df['order_delivered_customer_date'] -
    df['order_estimated_delivery_date']
).dt.days

delay = df.groupby('customer_unique_id').agg(
    avg_delivery_delay = ('delivery_delay_days', 'mean')
).reset_index()

print(f"✅ Delivery delay feature built: {delay.shape}")

# ============================================
# STEP 5 — COMBINE ALL FEATURES
# ============================================
features = rfm.merge(extra, on='customer_unique_id', how='left')
features = features.merge(delay, on='customer_unique_id', how='left')
features = features.merge(
    churn_labels[['customer_unique_id', 'churn']],
    on='customer_unique_id', how='left'
)

# Fill any nulls in delay
features['avg_delivery_delay'].fillna(0, inplace=True)

print(f"\n✅ All features combined: {features.shape}")
print(f"Columns: {list(features.columns)}")

# ============================================
# STEP 6 — ENCODE PAYMENT TYPE
# ============================================
features = pd.get_dummies(features,
                          columns=['preferred_payment'],
                          prefix='payment')

print(f"\n✅ Payment type encoded")
print(f"Final shape: {features.shape}")

# ============================================
# STEP 7 — SAVE FEATURE MATRIX
# ============================================
features.to_csv(SAVE_PATH + 'rfm_features.csv', index=False)
print(f"\n✅ RFM features saved to Drive")
print(features.head(3))

Phase 4 — XGBoost Churn Model

In [ ]:
# ============================================
# PHASE 4 — XGBOOST CHURN MODEL
# ============================================

import pandas as pd
import numpy as np
import os
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, f1_score,
                             precision_score, recall_score,
                             roc_auc_score, confusion_matrix,
                             classification_report)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ============================================
# STEP 1 — LOAD FEATURES
# ============================================
SAVE_PATH = '/content/drive/MyDrive/churnlens/data/processed/'
MODEL_PATH = '/content/drive/MyDrive/churnlens/models/'
os.makedirs(MODEL_PATH, exist_ok=True)

features = pd.read_csv(SAVE_PATH + 'rfm_features.csv')
print(f"✅ Features loaded: {features.shape}")

# ============================================
# STEP 2 — PREPARE X AND Y
# ============================================
X = features.drop(columns=[
    'customer_unique_id',
    'churn',
    'recency',      # removing — directly used in churn label
    'frequency'     # removing — directly used in churn label
])
y = features['churn']

print(f"\nFeatures (X): {X.shape}")
print(f"Target (y) distribution:\n{y.value_counts()}")

# ============================================
# STEP 3 — TRAIN TEST SPLIT
# ============================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\n✅ Train: {X_train.shape} | Test: {X_test.shape}")

# ============================================
# STEP 4 — HANDLE CLASS IMBALANCE WITH SMOTE
# ============================================
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f"\n✅ After SMOTE:")
print(f"Train shape: {X_train_sm.shape}")
print(f"Class distribution: {pd.Series(y_train_sm).value_counts().to_dict()}")

# ============================================
# STEP 5 — BASELINE DUMMY CLASSIFIER
# ============================================
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train_sm, y_train_sm)
y_dummy = dummy.predict(X_test)

print(f"\n--- Baseline Dummy Classifier ---")
print(f"Accuracy:  {accuracy_score(y_test, y_dummy):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_dummy, zero_division=0):.4f}")
print(f"AUC-ROC:   {roc_auc_score(y_test, y_dummy):.4f}")

# ============================================
# STEP 6 — TRAIN XGBOOST MODEL
# ============================================
xgb = XGBClassifier(
    n_estimators     = 200,
    max_depth        = 6,
    learning_rate    = 0.1,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    use_label_encoder= False,
    eval_metric      = 'logloss',
    random_state     = 42
)

xgb.fit(X_train_sm, y_train_sm)
y_pred     = xgb.predict(X_test)
y_pred_proba = xgb.predict_proba(X_test)[:, 1]

print(f"\n--- XGBoost Model Results ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"AUC-ROC:   {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred))

# ============================================
# STEP 7 — CONFUSION MATRIX PLOT
# ============================================
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=['Retained','Churned'],
            yticklabels=['Retained','Churned'])
plt.title('XGBoost Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(MODEL_PATH + 'confusion_matrix.png')
plt.show()

# ============================================
# STEP 8 — FEATURE IMPORTANCE PLOT
# ============================================
importance_df = pd.DataFrame({
    'feature'   : X.columns,
    'importance': xgb.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(8,5))
sns.barplot(data=importance_df, x='importance', y='feature',
            palette='YlOrRd_r')
plt.title('XGBoost Feature Importance')
plt.tight_layout()
plt.savefig(MODEL_PATH + 'feature_importance.png')
plt.show()

print(f"\nTop 5 features:")
print(importance_df.head(5))

# ============================================
# STEP 9 — SAVE MODEL + PREDICTIONS
# ============================================
# Save model
with open(MODEL_PATH + 'xgboost_churn_v1.pkl', 'wb') as f:
    pickle.dump(xgb, f)

# Save predictions with customer id
test_results = features.iloc[X_test.index][['customer_unique_id']].copy()
test_results['churn_actual']      = y_test.values
test_results['churn_predicted']   = y_pred
test_results['churn_probability'] = y_pred_proba
test_results.to_csv(SAVE_PATH + 'churn_predictions.csv', index=False)

print(f"\n✅ Model saved: xgboost_churn_v1.pkl")
print(f"✅ Predictions saved: churn_predictions.csv")

Phase 5 — NLP Review Integrity Detection

In [ ]:
# ============================================
# PHASE 5 — NLP REVIEW INTEGRITY DETECTION
# ============================================

import pandas as pd
import numpy as np
import os
import re
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report,
                             confusion_matrix)
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================
# STEP 1 — LOAD AMAZON REVIEWS
# ============================================
NLP_PATH   = '/content/drive/MyDrive/churnlens/data/nlp/'
SAVE_PATH  = '/content/drive/MyDrive/churnlens/data/processed/'
MODEL_PATH = '/content/drive/MyDrive/churnlens/models/'

# Load train file
amazon = pd.read_csv(NLP_PATH + 'train.csv',
                     header=None,
                     names=['rating', 'title', 'review'])

print(f"✅ Amazon reviews loaded: {amazon.shape}")
print(f"\nRating distribution:\n{amazon['rating'].value_counts()}")
print(f"\nSample:\n{amazon.head(3)}")

In [ ]:
# ============================================
# STEP 2 — PREPROCESS + MISMATCH LABEL
# ============================================

# Sample 500k for faster training — still huge
amazon_sample = amazon.sample(500000, random_state=42).reset_index(drop=True)
print(f"✅ Sampled: {amazon_sample.shape}")

# Clean text function
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)       # remove URLs
    text = re.sub(r'[^a-z\s]', '', text)      # keep only letters
    text = re.sub(r'\s+', ' ', text).strip()  # remove extra spaces
    return text

# Apply cleaning
amazon_sample['clean_review'] = amazon_sample['review'].apply(clean_text)
amazon_sample['clean_title']  = amazon_sample['title'].apply(clean_text)

# Combine title + review for richer text
amazon_sample['full_text'] = (
    amazon_sample['clean_title'] + ' ' +
    amazon_sample['clean_review']
)

print(f"✅ Text cleaned")

# ============================================
# VADER SENTIMENT SCORE
# ============================================
analyzer = SentimentIntensityAnalyzer()

def get_vader_sentiment(text):
    score = analyzer.polarity_scores(text)['compound']
    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

print("Running VADER on 500k reviews — takes 2-3 mins...")
amazon_sample['vader_sentiment'] = amazon_sample['full_text'].apply(
    get_vader_sentiment
)
print(f"✅ VADER done")
print(f"Sentiment distribution:\n{amazon_sample['vader_sentiment'].value_counts()}")

# ============================================
# MISMATCH LABEL LOGIC
# ============================================
# rating 2 = positive, rating 1 = negative (polarity dataset)
# mismatch = rating says positive but vader says negative OR vice versa

def create_mismatch_label(row):
    rating_sentiment = 'positive' if row['rating'] == 2 else 'negative'
    if rating_sentiment == 'positive' and row['vader_sentiment'] == 'negative':
        return 1  # MISMATCH
    elif rating_sentiment == 'negative' and row['vader_sentiment'] == 'positive':
        return 1  # MISMATCH
    else:
        return 0  # CONSISTENT

amazon_sample['mismatch'] = amazon_sample.apply(
    create_mismatch_label, axis=1
)

print(f"\nMismatch distribution:")
print(amazon_sample['mismatch'].value_counts())
print(f"Mismatch %: {amazon_sample['mismatch'].mean()*100:.1f}%")

In [ ]:
# ============================================
# STEP 3 — TFIDF + LOGISTIC REGRESSION
# ============================================

# ============================================
# TRAIN TEST SPLIT
# ============================================
X = amazon_sample['full_text']
y = amazon_sample['mismatch']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"✅ Train: {X_train.shape} | Test: {X_test.shape}")

# ============================================
# TFIDF VECTORIZER
# ============================================
print("\nFitting TF-IDF — takes 1-2 mins...")
tfidf = TfidfVectorizer(
    max_features = 50000,   # top 50k words
    ngram_range  = (1, 2),  # unigrams + bigrams
    min_df       = 5,       # ignore rare words
    max_df       = 0.95,    # ignore too common words
    sublinear_tf = True     # apply log normalization
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)
print(f"✅ TF-IDF done: {X_train_tfidf.shape}")

# ============================================
# TRAIN LOGISTIC REGRESSION
# ============================================
print("\nTraining Logistic Regression...")
lr = LogisticRegression(
    max_iter     = 1000,
    class_weight = 'balanced',
    random_state = 42,
    C            = 1.0
)
lr.fit(X_train_tfidf, y_train)
print("✅ Model trained")

# ============================================
# EVALUATE
# ============================================
y_pred       = lr.predict(X_test_tfidf)
y_pred_proba = lr.predict_proba(X_test_tfidf)[:, 1]

print(f"\n--- Logistic Regression Results ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

# ============================================
# CONFUSION MATRIX
# ============================================
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Consistent','Mismatch'],
            yticklabels=['Consistent','Mismatch'])
plt.title('NLP Model — Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(MODEL_PATH + 'nlp_confusion_matrix.png')
plt.show()

# ============================================
# SAVE TFIDF + MODEL
# ============================================
with open(MODEL_PATH + 'tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

with open(MODEL_PATH + 'lr_mismatch_v1.pkl', 'wb') as f:
    pickle.dump(lr, f)

print(f"\n✅ TF-IDF vectorizer saved")
print(f"✅ Logistic Regression model saved")

In [ ]:
# ============================================
# STEP 4 — APPLY NLP MODEL ON OLIST REVIEWS
# ============================================

import time

# Install deep-translator if not already
import subprocess
subprocess.run(['pip', 'install', 'deep-translator', '-q'])

from deep_translator import GoogleTranslator

SAVE_PATH  = '/content/drive/MyDrive/churnlens/data/processed/'
MODEL_PATH = '/content/drive/MyDrive/churnlens/models/'

# ============================================
# LOAD OLIST CLEANED DATA
# ============================================
df = pd.read_csv(SAVE_PATH + 'olist_cleaned.csv')

# Keep only rows with real review text
reviews_df = df[
    df['review_comment_message'] != 'no review'
][['customer_unique_id', 'review_score',
   'review_comment_message']].copy()

reviews_df = reviews_df.drop_duplicates(
    subset='review_comment_message'
).reset_index(drop=True)

print(f"✅ Olist reviews with text: {reviews_df.shape}")
print(reviews_df.head(3))

# ============================================
# TRANSLATE PORTUGUESE TO ENGLISH
# (sample 5000 to avoid API limits)
# ============================================
reviews_sample = reviews_df.sample(
    min(5000, len(reviews_df)),
    random_state=42
).reset_index(drop=True)

print(f"\nTranslating {len(reviews_sample)} reviews...")
print("This takes 3-5 mins — please wait...")

def translate_text(text):
    try:
        return GoogleTranslator(
            source='auto', target='en'
        ).translate(str(text)[:500])
    except:
        return str(text)

translated = []
for i, text in enumerate(reviews_sample['review_comment_message']):
    translated.append(translate_text(text))
    if (i+1) % 500 == 0:
        print(f"  Translated {i+1}/{len(reviews_sample)}...")
        time.sleep(1)  # avoid rate limiting

reviews_sample['review_english'] = translated
print(f"\n✅ Translation done")
print(reviews_sample[['review_comment_message',
                       'review_english']].head(3))

In [ ]:
# ============================================
# STEP 5 — APPLY NLP MODEL ON OLIST REVIEWS
# ============================================

# Load saved models
with open(MODEL_PATH + 'tfidf_vectorizer.pkl', 'rb') as f:
    tfidf = pickle.load(f)

with open(MODEL_PATH + 'lr_mismatch_v1.pkl', 'rb') as f:
    lr = pickle.load(f)

# ============================================
# CLEAN TRANSLATED TEXT
# ============================================
reviews_sample['clean_english'] = reviews_sample[
    'review_english'
].apply(clean_text)

# ============================================
# VADER SENTIMENT ON OLIST
# ============================================
reviews_sample['vader_sentiment'] = reviews_sample[
    'clean_english'
].apply(get_vader_sentiment)

# ============================================
# TFIDF + LR MISMATCH PREDICTION
# ============================================
X_olist       = tfidf.transform(reviews_sample['clean_english'])
mismatch_pred = lr.predict(X_olist)
mismatch_prob = lr.predict_proba(X_olist)[:, 1]

reviews_sample['mismatch_flag']  = mismatch_pred
reviews_sample['integrity_score'] = mismatch_prob

# ============================================
# REVIEW SCORE MAPPING
# ============================================
# Map 1-5 star to sentiment
def map_star_sentiment(score):
    if score >= 4:
        return 'positive'
    elif score <= 2:
        return 'negative'
    else:
        return 'neutral'

reviews_sample['star_sentiment'] = reviews_sample[
    'review_score'
].apply(map_star_sentiment)

# ============================================
# FINAL INTEGRITY FLAG
# ============================================
# Combine LR prediction + VADER + star rating
def final_integrity(row):
    if row['mismatch_flag'] == 1:
        return 1
    if (row['star_sentiment'] == 'positive' and
            row['vader_sentiment'] == 'negative'):
        return 1
    if (row['star_sentiment'] == 'negative' and
            row['vader_sentiment'] == 'positive'):
        return 1
    return 0

reviews_sample['final_mismatch'] = reviews_sample.apply(
    final_integrity, axis=1
)

print(f"✅ NLP applied on Olist reviews")
print(f"\nMismatch distribution:")
print(reviews_sample['final_mismatch'].value_counts())
print(f"Mismatch %: {reviews_sample['final_mismatch'].mean()*100:.1f}%")

# ============================================
# AGGREGATE INTEGRITY SCORE PER CUSTOMER
# ============================================
customer_integrity = reviews_sample.groupby(
    'customer_unique_id'
).agg(
    avg_integrity_score = ('integrity_score', 'mean'),
    has_mismatch        = ('final_mismatch', 'max')
).reset_index()

print(f"\n✅ Customer integrity scores: {customer_integrity.shape}")
print(customer_integrity.head(5))

# Save
customer_integrity.to_csv(
    SAVE_PATH + 'customer_integrity.csv', index=False
)
print(f"\n✅ Integrity scores saved to Drive")

Phase 6 — Final Risk Score

In [ ]:
# ============================================
# PHASE 6 — FINAL RISK SCORE CALCULATION
# ============================================

import pandas as pd
import numpy as np
import os
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

SAVE_PATH  = '/content/drive/MyDrive/churnlens/data/processed/'
MODEL_PATH = '/content/drive/MyDrive/churnlens/models/'

# ============================================
# STEP 1 — LOAD CHURN + INTEGRITY SCORES
# ============================================
churn_preds = pd.read_csv(SAVE_PATH + 'churn_predictions_full.csv')
integrity   = pd.read_csv(SAVE_PATH + 'customer_integrity.csv')
features    = pd.read_csv(SAVE_PATH + 'rfm_features.csv')

print(f"✅ Churn predictions: {churn_preds.shape}")
print(f"✅ Integrity scores:  {integrity.shape}")
print(f"✅ Features:          {features.shape}")

# ============================================
# STEP 2 — MERGE ALL SCORES
# ============================================
# Start with all customers from features
final_df = features[[
    'customer_unique_id', 'churn',
    'monetary', 'avg_review_score',
    'total_categories', 'avg_delivery_delay'
]].copy()

# Merge churn probability
final_df = final_df.merge(
    churn_preds[['customer_unique_id', 'churn_probability']],
    on='customer_unique_id', how='left'
)

# Merge integrity score
final_df = final_df.merge(
    integrity[['customer_unique_id', 'avg_integrity_score', 'has_mismatch']],
    on='customer_unique_id', how='left'
)

# Fill customers with no reviews — integrity = 0 (no mismatch detected)
final_df['avg_integrity_score'].fillna(0, inplace=True)
final_df['has_mismatch'].fillna(0, inplace=True)

# Fill customers not in test set — use median churn probability
median_churn = final_df['churn_probability'].median()
final_df['churn_probability'].fillna(median_churn, inplace=True)

print(f"\n✅ Merged: {final_df.shape}")
print(f"Null check:\n{final_df.isnull().sum()}")

# ============================================
# STEP 3 — CALCULATE FINAL RISK SCORE
# ============================================
# Formula: risk = 0.6 * churn + 0.4 * integrity
final_df['risk_score'] = (
    0.6 * final_df['churn_probability'] +
    0.4 * final_df['avg_integrity_score']
)

# ============================================
# STEP 4 — RISK CATEGORY
# ============================================
def categorize_risk(score):
    if score > 0.7:
        return 'HIGH'
    elif score >= 0.4:
        return 'MEDIUM'
    else:
        return 'LOW'

final_df['risk_category'] = final_df['risk_score'].apply(categorize_risk)

print(f"\n✅ Risk scores calculated")
print(f"\nRisk category distribution:")
print(final_df['risk_category'].value_counts())
print(f"\nRisk % breakdown:")
print(final_df['risk_category'].value_counts(normalize=True) * 100)

# ============================================
# STEP 5 — VISUALIZE RISK DISTRIBUTION
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Risk category pie chart
risk_counts = final_df['risk_category'].value_counts()
colors = ['#ff6b47', '#ffc400', '#47ff96']
axes[0].pie(risk_counts, labels=risk_counts.index,
            autopct='%1.1f%%', colors=colors,
            startangle=90)
axes[0].set_title('Customer Risk Distribution')

# Risk score histogram
axes[1].hist(final_df['risk_score'], bins=50,
             color='#47c4ff', edgecolor='white', alpha=0.8)
axes[1].set_title('Risk Score Distribution')
axes[1].set_xlabel('Risk Score')
axes[1].set_ylabel('Number of Customers')

plt.tight_layout()
plt.savefig(MODEL_PATH + 'risk_distribution.png')
plt.show()

# ============================================
# STEP 6 — SAVE FINAL DATASET
# ============================================
final_df.to_csv(SAVE_PATH + 'final_risk_scores.csv', index=False)
print(f"\n✅ Final risk scores saved")
print(f"\nSample output:")
print(final_df[['customer_unique_id', 'churn_probability',
                'avg_integrity_score', 'risk_score',
                'risk_category']].head(10))

In [ ]:
# ============================================
# FIX — PREDICT CHURN FOR ALL 96096 CUSTOMERS
# ============================================

SAVE_PATH  = '/content/drive/MyDrive/churnlens/data/processed/'
MODEL_PATH = '/content/drive/MyDrive/churnlens/models/'

# Load model and full features
with open(MODEL_PATH + 'xgboost_churn_v1.pkl', 'rb') as f:
    xgb = pickle.load(f)

features = pd.read_csv(SAVE_PATH + 'rfm_features.csv')

# Same columns used in training
X_all = features.drop(columns=[
    'customer_unique_id',
    'churn',
    'recency',
    'frequency'
])

# Predict on ALL customers
all_churn_proba = xgb.predict_proba(X_all)[:, 1]
all_churn_pred  = xgb.predict(X_all)

# Save full predictions
full_predictions = pd.DataFrame({
    'customer_unique_id' : features['customer_unique_id'],
    'churn_actual'       : features['churn'],
    'churn_predicted'    : all_churn_pred,
    'churn_probability'  : all_churn_proba
})

full_predictions.to_csv(
    SAVE_PATH + 'churn_predictions_full.csv', index=False
)

print(f"✅ Full predictions saved: {full_predictions.shape}")
print(f"\nChurn probability stats:")
print(full_predictions['churn_probability'].describe())
print(f"\nSample:")
print(full_predictions.head(5))